# Step 6 — Generate Embeddings

Pada tahap ini kita akan:

1. Membaca seluruh data dari tabel `chunks`.
2. Membuat embedding menggunakan model `BAAI/bge-small-en-v1.5`.
3. Menyimpan embedding ke tabel `embeddings`.

> Pastikan package `sentence-transformers` sudah terpasang.


In [1]:
from pathlib import Path
import sqlite3

DB_PATH = Path("../data/linux_docs.db")
assert DB_PATH.exists(), "Database tidak ditemukan."

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cursor = conn.cursor()


## Membuat tabel `embeddings`

In [2]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS embeddings (
    chunk_id INTEGER PRIMARY KEY,
    model TEXT NOT NULL,
    dimension INTEGER NOT NULL,
    embedding BLOB NOT NULL,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY(chunk_id) REFERENCES chunks(id)
);
''')

conn.commit()
print("✅ Tabel embeddings siap.")


✅ Tabel embeddings siap.


## Load model embedding

In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np

MODEL_NAME = "BAAI/bge-small-en-v1.5"

model = SentenceTransformer(MODEL_NAME)
print("Model loaded:", MODEL_NAME)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded: BAAI/bge-small-en-v1.5


## Membaca chunks

In [4]:
cursor.execute('''
SELECT id, content
FROM chunks
ORDER BY id;
''')

chunks = cursor.fetchall()
print(f"Total chunks: {len(chunks)}")


Total chunks: 41


## Generate embedding dan simpan

In [5]:
cursor.execute("DELETE FROM embeddings")

for row in chunks:
    vector = model.encode(
        row["content"],
        normalize_embeddings=True
    )

    vector = np.asarray(vector, dtype=np.float32)

    cursor.execute(
        '''
        INSERT INTO embeddings(
            chunk_id,
            model,
            dimension,
            embedding
        )
        VALUES (?, ?, ?, ?)
        ''',
        (
            row["id"],
            MODEL_NAME,
            vector.shape[0],
            vector.tobytes()
        )
    )

conn.commit()

print("✅ Embeddings berhasil disimpan.")


✅ Embeddings berhasil disimpan.


## Verifikasi

In [6]:
cursor.execute('''
SELECT
    chunk_id,
    model,
    dimension,
    length(embedding) AS bytes
FROM embeddings
LIMIT 5;
''')

for row in cursor.fetchall():
    print(dict(row))


{'chunk_id': 42, 'model': 'BAAI/bge-small-en-v1.5', 'dimension': 384, 'bytes': 1536}
{'chunk_id': 43, 'model': 'BAAI/bge-small-en-v1.5', 'dimension': 384, 'bytes': 1536}
{'chunk_id': 44, 'model': 'BAAI/bge-small-en-v1.5', 'dimension': 384, 'bytes': 1536}
{'chunk_id': 45, 'model': 'BAAI/bge-small-en-v1.5', 'dimension': 384, 'bytes': 1536}
{'chunk_id': 46, 'model': 'BAAI/bge-small-en-v1.5', 'dimension': 384, 'bytes': 1536}


## Catatan

Embedding disimpan sebagai **BLOB float32**.

Keuntungan pendekatan ini:

- Database tetap hanya satu file SQLite.
- Mudah diekspor ke Qdrant, pgvector, atau sistem lain di masa depan.
- Pada Step 7 kita akan membaca kembali BLOB tersebut dan melakukan **semantic search** menggunakan cosine similarity.


In [7]:
conn.close()
print("Selesai 🎉")

Selesai 🎉
